# 23. Multimodal RAG with ColPali

Skip captioning: embed page images directly with a late-interaction vision retriever.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Captioning compresses an image into a few sentences and loses detail. **ColPali** skips OCR and captioning: a vision-language model embeds a page image into ~1,000 patch vectors and matches them with query-token vectors using **late interaction** (MaxSim, as in ColBERT). Layout, tables, charts and fonts all contribute to retrieval.

## How it works

1. Render each page as an image
2. Embed with ColPali -> a matrix of patch vectors per page
3. Embed the query -> a matrix of token vectors
4. Score = for each query token take the best-matching patch, then sum (MaxSim)
5. Show the best page image or feed it to a vision LLM

```text
 page image --> ColPali --> [~1024 patch vectors]
 query      --> ColPali --> [n token vectors]
 score(query, page) = sum over query tokens of max over patches of dot(token, patch)
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install colpali-engine torch pillow` (GPU or Apple Silicon recommended; downloads ~6 GB)

ColPali embeds a page *image* into many patch vectors and matches them to query tokens (ColBERT-style late interaction). No OCR or captioning step, so layout, tables and charts are preserved.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import torch
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model_name = "vidore/colpali-v1.3"
model = ColPali.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map=device).eval()
processor = ColPaliProcessor.from_pretrained(model_name)
print("loaded on", device)

In [ ]:
paths = make_figures()
images = [Image.open(p).convert("RGB") for p in paths]

image_embeddings = []
for i in range(0, len(images), 2):                      # small batches to save memory
    batch = processor.process_images(images[i:i + 2]).to(model.device)
    with torch.no_grad():
        image_embeddings.extend(list(torch.unbind(model(**batch).to("cpu"))))
print(len(image_embeddings), "page embeddings")

Score the query against every page with `score_multi_vector` (MaxSim over patches).

In [ ]:
queries = ["Which chunk size gives the best hit rate?", "Which pipeline stage is the slowest?"]
batch_q = processor.process_queries(queries).to(model.device)
with torch.no_grad():
    query_embeddings = list(torch.unbind(model(**batch_q).to("cpu")))

scores = processor.score_multi_vector(query_embeddings, image_embeddings)
for q, row in zip(queries, scores):
    best = int(row.argmax())
    print(f"{q}\n  -> {paths[best].name} (score {float(row[best]):.1f})")

In [ ]:
from IPython.display import display
display(images[int(scores[0].argmax())])

Feed the retrieved page image to a vision LLM (e.g. `llava` through Ollama) to generate the final answer.

## Memory and hardware notes
ColPali v1.3 is a ~3B parameter model. It needs a GPU or Apple Silicon with ~8 GB+ free memory. On smaller machines run it in Colab or use captioning (notebook 22).

## Understanding MaxSim by hand
This tiny numpy example reproduces what `score_multi_vector` does.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
q = rng.normal(size=(4, 8))            # 4 query tokens, 8 dims
page = rng.normal(size=(20, 8))        # 20 patches
sim = q @ page.T                       # (4 tokens x 20 patches)
print("best patch for each query token:", sim.argmax(axis=1))
print("MaxSim score:", float(sim.max(axis=1).sum()))

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| No OCR, parsing or captioning pipeline | Large model and storage (many vectors per page) |
| Understands layout, tables and charts | Slower search than single-vector retrieval |
| State-of-the-art on visual document retrieval | Needs a vision LLM to read the retrieved page |

## When to use it

Scanned PDFs, slides, invoices and reports where layout carries meaning.

## Experiments to try

- Index a real multi-page PDF (render pages with `pdf2image`).
- Compare retrieval against caption-based RAG on chart questions.
- Pass the top page to a vision LLM for the final answer.

## Key takeaways

- Treat pages as images.
- Late interaction keeps fine-grained matching.
- Storage cost is the trade-off.

---
**Next:** [24. Retrieval with a Feedback Loop](../24_retrieval_with_feedback_loop/retrieval_with_feedback_loop.ipynb)